In [1]:
!pip -q install datasets

import re
import math
import csv
import random
from collections import Counter
from datasets import load_dataset

In [2]:
def tokenize(text):
    # Supports Indic Unicode scripts such as Devanagari, Tamil, Telugu, Urdu, etc.
    return re.findall(r"\w+", str(text).lower(), flags=re.UNICODE)

In [3]:
# 23 Indic languages in IndicCorpV2.
# English (en.txt) is intentionally excluded.

LANGUAGES = {
    "Assamese": "as.txt",
    "Bodo": "bd.txt",
    "Bengali": "bn.txt",
    "Dogri": "dg.txt",
    "Konkani": "gom.txt",
    "Gujarati": "gu.txt",
    "Hindi": "hi-1.txt",
    "Khasi": "kha.txt",
    "Kannada": "kn.txt",
    "Kashmiri": "ks.txt",
    "Maithili": "mai.txt",
    "Malayalam": "ml.txt",
    "Manipuri": "mni.txt",
    "Marathi": "mr.txt",
    "Nepali": "ne.txt",
    "Odia": "or.txt",
    "Punjabi": "pa.txt",
    "Sanskrit": "sa.txt",
    "Santali": "sat.txt",
    "Sindhi": "sd.txt",
    "Tamil": "ta.txt",
    "Telugu": "te.txt",
    "Urdu": "ur.txt"
}

SAMPLES_PER_LANGUAGE = 1000

print("Number of language classes:", len(LANGUAGES))
print("Total texts required:", len(LANGUAGES) * SAMPLES_PER_LANGUAGE)

Number of language classes: 23
Total texts required: 23000


In [4]:
documents_by_label = {}

BASE_URL = (
    "https://huggingface.co/datasets/ai4bharat/IndicCorpV2/"
    "resolve/main/data/"
)

for language_name, file_name in LANGUAGES.items():
    print(f"Loading {language_name}...")

    file_url = BASE_URL + file_name

    # The Hugging Face "text" loader reads the remote .txt file line by line.
    stream = load_dataset(
        "text",
        data_files=file_url,
        split="train",
        streaming=True
    )

    language_documents = []

    for record in stream:
        text = record["text"].strip()

        if len(tokenize(text)) >= 3:
            language_documents.append(text)

        if len(language_documents) == SAMPLES_PER_LANGUAGE:
            break

    if len(language_documents) < SAMPLES_PER_LANGUAGE:
        raise ValueError(
            f"Only {len(language_documents)} texts collected for {language_name}"
        )

    documents_by_label[language_name] = language_documents
    print(f"Collected {len(language_documents)} texts")

print("\nTotal texts collected:", sum(
    len(texts) for texts in documents_by_label.values()
))

Loading Assamese...
Collected 1000 texts
Loading Bodo...
Collected 1000 texts
Loading Bengali...
Collected 1000 texts
Loading Dogri...
Collected 1000 texts
Loading Konkani...
Collected 1000 texts
Loading Gujarati...
Collected 1000 texts
Loading Hindi...
Collected 1000 texts
Loading Khasi...
Collected 1000 texts
Loading Kannada...
Collected 1000 texts
Loading Kashmiri...
Collected 1000 texts
Loading Maithili...
Collected 1000 texts
Loading Malayalam...
Collected 1000 texts
Loading Manipuri...
Collected 1000 texts
Loading Marathi...
Collected 1000 texts
Loading Nepali...
Collected 1000 texts
Loading Odia...
Collected 1000 texts
Loading Punjabi...
Collected 1000 texts
Loading Sanskrit...
Collected 1000 texts
Loading Santali...
Collected 1000 texts
Loading Sindhi...
Collected 1000 texts
Loading Tamil...
Collected 1000 texts
Loading Telugu...
Collected 1000 texts
Loading Urdu...
Collected 1000 texts

Total texts collected: 23000


In [5]:
TRAIN_PER_LANGUAGE = 750
TEST_PER_LANGUAGE = 250

random_generator = random.Random(42)

train_texts = []
train_labels = []

test_texts = []
test_labels = []

for language_name, texts in documents_by_label.items():
    texts = texts.copy()
    random_generator.shuffle(texts)

    for text in texts[:TRAIN_PER_LANGUAGE]:
        train_texts.append(text)
        train_labels.append(language_name)

    for text in texts[TRAIN_PER_LANGUAGE:]:
        test_texts.append(text)
        test_labels.append(language_name)

print("Training texts:", len(train_texts))
print("Testing texts:", len(test_texts))
print("Classes:", len(set(train_labels)))

Training texts: 17250
Testing texts: 5750
Classes: 23


In [6]:
train_tokens = [tokenize(text) for text in train_texts]
test_tokens = [tokenize(text) for text in test_texts]

print("Example tokenized text:")
print(train_tokens[0][:20])

Example tokenized text:
['১', 'হ', 'জ', 'ৰ', 'টক']


In [7]:
document_frequency = Counter()

for document in train_tokens:
    document_frequency.update(set(document))

# Limit vocabulary only to keep pure-Python training practical in Colab.
# All 23,000 texts are still used.
MAX_VOCABULARY_SIZE = 8000

selected_words = sorted(
    document_frequency.items(),
    key=lambda item: (-item[1], item[0])
)[:MAX_VOCABULARY_SIZE]

vocabulary = {
    word: index
    for index, (word, frequency) in enumerate(selected_words)
}

df_values = [
    document_frequency[word]
    for word, index in vocabulary.items()
]

N = len(train_tokens)
df_max = max(df_values)

print("Training documents:", N)
print("Vocabulary size:", len(vocabulary))
print("Maximum document frequency:", df_max)

Training documents: 17250
Vocabulary size: 8000
Maximum document frequency: 4561


In [8]:
def create_tf_vector(tokens, vocabulary, tf_type):
    """
    raw:
        TF(t, d) = count(t, d)

    length_normalized:
        TF(t, d) = count(t, d) / total number of words in d

    max_frequency_normalized:
        TF(t, d) = count(t, d) / maximum word frequency in d
    """

    counts = Counter(tokens)
    vector = {}

    if tf_type == "raw":
        denominator = 1

    elif tf_type == "length_normalized":
        denominator = len(tokens)

    elif tf_type == "max_frequency_normalized":
        denominator = max(counts.values())

    else:
        raise ValueError("Invalid TF type")

    for word, count in counts.items():
        if word in vocabulary:
            word_index = vocabulary[word]
            vector[word_index] = count / denominator

    return vector

In [9]:
def create_idf_vector(df_values, N, df_max, idf_type):
    """
    unnormalized:
        IDF(t) = ln(N / df(t))

    normalized:
        IDF(t) = ln(df_max / df(t))

    df_max is the document frequency of the word appearing
    in the maximum number of training documents.
    """

    idf_values = []

    for df in df_values:
        if idf_type == "unnormalized":
            idf = math.log(N / df)

        elif idf_type == "normalized":
            idf = math.log(df_max / df)

        else:
            raise ValueError("Invalid IDF type")

        idf_values.append(idf)

    return idf_values

In [10]:
def create_tfidf_vectors(tokenized_documents, vocabulary, tf_type, idf_values):
    tfidf_vectors = []

    for tokens in tokenized_documents:
        tf_vector = create_tf_vector(tokens, vocabulary, tf_type)

        tfidf_vector = {}

        for word_index, tf_value in tf_vector.items():
            tfidf_value = tf_value * idf_values[word_index]

            # Values of zero do not help the classifier
            if tfidf_value != 0:
                tfidf_vector[word_index] = tfidf_value

        tfidf_vectors.append(tfidf_vector)

    return tfidf_vectors

In [11]:
class_names = sorted(LANGUAGES.keys())

label_to_index = {
    label: index
    for index, label in enumerate(class_names)
}

index_to_label = {
    index: label
    for label, index in label_to_index.items()
}

y_train = [label_to_index[label] for label in train_labels]
y_test = [label_to_index[label] for label in test_labels]

NUMBER_OF_CLASSES = len(class_names)
NUMBER_OF_FEATURES = len(vocabulary)

print("Classes:", NUMBER_OF_CLASSES)
print("Features:", NUMBER_OF_FEATURES)

Classes: 23
Features: 8000


In [12]:
def softmax(scores):
    maximum_score = max(scores)

    exponentials = [
        math.exp(score - maximum_score)
        for score in scores
    ]

    total = sum(exponentials)

    return [
        value / total
        for value in exponentials
    ]


def train_logistic_regression(
    X_train,
    y_train,
    number_of_classes,
    number_of_features,
    epochs=4,
    learning_rate=0.10
):
    """
    Multiclass Logistic Regression trained using softmax
    and stochastic gradient descent.

    No sklearn model is used.
    """

    weights = [
        [0.0] * number_of_features
        for _ in range(number_of_classes)
    ]

    biases = [0.0] * number_of_classes

    indices = list(range(len(X_train)))

    for epoch in range(epochs):
        random.Random(42 + epoch).shuffle(indices)

        current_learning_rate = learning_rate / (1 + 0.20 * epoch)

        for row in indices:
            features = X_train[row]
            actual_class = y_train[row]

            scores = []

            for class_index in range(number_of_classes):
                score = biases[class_index]

                for feature_index, feature_value in features.items():
                    score += weights[class_index][feature_index] * feature_value

                scores.append(score)

            probabilities = softmax(scores)

            # Gradient descent update
            for class_index in range(number_of_classes):
                target = 1.0 if class_index == actual_class else 0.0
                error = probabilities[class_index] - target

                biases[class_index] -= current_learning_rate * error

                for feature_index, feature_value in features.items():
                    weights[class_index][feature_index] -= (
                        current_learning_rate * error * feature_value
                    )

        print(f"Epoch {epoch + 1}/{epochs} completed")

    return weights, biases

In [13]:
def predict_logistic_regression(X_test, weights, biases):
    predictions = []

    number_of_classes = len(biases)

    for features in X_test:
        best_class = 0
        best_score = float("-inf")

        for class_index in range(number_of_classes):
            score = biases[class_index]

            for feature_index, feature_value in features.items():
                score += weights[class_index][feature_index] * feature_value

            if score > best_score:
                best_score = score
                best_class = class_index

        predictions.append(best_class)

    return predictions

In [14]:
def calculate_metrics(y_true, y_predicted, number_of_classes):
    correct_predictions = 0

    true_positive = [0] * number_of_classes
    false_positive = [0] * number_of_classes
    false_negative = [0] * number_of_classes
    support = [0] * number_of_classes

    for actual, predicted in zip(y_true, y_predicted):
        support[actual] += 1

        if actual == predicted:
            correct_predictions += 1
            true_positive[actual] += 1
        else:
            false_positive[predicted] += 1
            false_negative[actual] += 1

    accuracy = correct_predictions / len(y_true)

    weighted_precision = 0
    weighted_recall = 0
    weighted_f1 = 0

    for class_index in range(number_of_classes):
        tp = true_positive[class_index]
        fp = false_positive[class_index]
        fn = false_negative[class_index]

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        f1_score = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0
        )

        weight = support[class_index] / len(y_true)

        weighted_precision += precision * weight
        weighted_recall += recall * weight
        weighted_f1 += f1_score * weight

    return {
        "accuracy": accuracy,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1
    }

In [15]:
configurations = [
    ("Model 1: Raw TF + Unnormalized IDF",
     "raw", "unnormalized"),

    ("Model 2: Raw TF + Normalized IDF",
     "raw", "normalized"),

    ("Model 3: Length-Normalized TF + Unnormalized IDF",
     "length_normalized", "unnormalized"),

    ("Model 4: Length-Normalized TF + Normalized IDF",
     "length_normalized", "normalized"),

    ("Model 5: Max-Frequency-Normalized TF + Unnormalized IDF",
     "max_frequency_normalized", "unnormalized"),

    ("Model 6: Max-Frequency-Normalized TF + Normalised IDF",
     "max_frequency_normalized", "normalized")
]

results = []

for model_name, tf_type, idf_type in configurations:
    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    idf_values = create_idf_vector(
        df_values,
        N,
        df_max,
        idf_type
    )

    X_train_tfidf = create_tfidf_vectors(
        train_tokens,
        vocabulary,
        tf_type,
        idf_values
    )

    X_test_tfidf = create_tfidf_vectors(
        test_tokens,
        vocabulary,
        tf_type,
        idf_values
    )

    weights, biases = train_logistic_regression(
        X_train_tfidf,
        y_train,
        NUMBER_OF_CLASSES,
        NUMBER_OF_FEATURES,
        epochs=4,
        learning_rate=0.10
    )

    predictions = predict_logistic_regression(
        X_test_tfidf,
        weights,
        biases
    )

    metrics = calculate_metrics(
        y_test,
        predictions,
        NUMBER_OF_CLASSES
    )

    results.append({
        "Model": model_name,
        "TF": tf_type,
        "IDF": idf_type,
        "Accuracy": metrics["accuracy"],
        "Weighted Precision": metrics["weighted_precision"],
        "Weighted Recall": metrics["weighted_recall"],
        "Weighted F1 Score": metrics["weighted_f1"]
    })


Model 1: Raw TF + Unnormalized IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed

Model 2: Raw TF + Normalized IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed

Model 3: Length-Normalized TF + Unnormalized IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed

Model 4: Length-Normalized TF + Normalized IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed

Model 5: Max-Frequency-Normalized TF + Unnormalized IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed

Model 6: Max-Frequency-Normalized TF + Normalised IDF
Epoch 1/4 completed
Epoch 2/4 completed
Epoch 3/4 completed
Epoch 4/4 completed


In [16]:
results = sorted(
    results,
    key=lambda row: row["Weighted F1 Score"],
    reverse=True
)

print(
    f"{'Rank':<5} {'Model':<62} "
    f"{'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1 Score':<10}"
)

print("-" * 120)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank:<5} "
        f"{result['Model']:<62} "
        f"{result['Accuracy']:<10.4f} "
        f"{result['Weighted Precision']:<10.4f} "
        f"{result['Weighted Recall']:<10.4f} "
        f"{result['Weighted F1 Score']:<10.4f}"
    )

Rank  Model                                                          Accuracy   Precision  Recall     F1 Score  
------------------------------------------------------------------------------------------------------------------------
1     Model 6: Max-Frequency-Normalized TF + Normalised IDF          0.9099     0.9127     0.9099     0.9105    
2     Model 5: Max-Frequency-Normalized TF + Unnormalized IDF        0.9099     0.9107     0.9099     0.9097    
3     Model 1: Raw TF + Unnormalized IDF                             0.9009     0.9022     0.9009     0.8998    
4     Model 2: Raw TF + Normalized IDF                               0.8972     0.8980     0.8972     0.8974    
5     Model 3: Length-Normalized TF + Unnormalized IDF               0.8988     0.9027     0.8988     0.8968    
6     Model 4: Length-Normalized TF + Normalized IDF                 0.8908     0.9079     0.8908     0.8926    


In [17]:
with open("six_model_tfidf_comparison.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print("Saved: six_model_tfidf_comparison.csv")

Saved: six_model_tfidf_comparison.csv
